In [1]:
##检查原始dcm文件的类型
import os
import pydicom

folder_path = r"G:\wxy_codes\Fetal_hydronephrosis\前期数据20260320\图像\阳性数据\2齐红月之子-孙铭晨\202411101170\0_JPG"  # 改成你的实际路径

dcm_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(".dcm")])

print(f"共找到 {len(dcm_files)} 个 DICOM 文件\n")

for file_name in dcm_files:
    dcm_path = os.path.join(folder_path, file_name)
    print("=" * 80)
    print("文件名:", file_name)

    try:
        ds = pydicom.dcmread(dcm_path)

        # 基本信息
        print("PatientID:", getattr(ds, "PatientID", "无"))
        print("Modality:", getattr(ds, "Modality", "无"))
        print("PhotometricInterpretation:", getattr(ds, "PhotometricInterpretation", "无"))
        print("Rows:", getattr(ds, "Rows", "无"))
        print("Columns:", getattr(ds, "Columns", "无"))
        print("NumberOfFrames:", getattr(ds, "NumberOfFrames", "默认单帧"))

        # 像素数组
        img = ds.pixel_array
        print("pixel_array type:", type(img))
        print("pixel_array shape:", img.shape)
        print("pixel_array dtype:", img.dtype)

        # 判断单帧 / 多帧
        if img.ndim == 2:
            print("判断结果: 单帧灰度图")
        elif img.ndim == 3:
            if img.shape[-1] in [3, 4]:
                print("判断结果: 单帧彩色图")
            else:
                print("判断结果: 多帧图像")
        elif img.ndim > 3:
            print("判断结果: 高维多帧/特殊格式图像")
        else:
            print("判断结果: 未知格式")

    except Exception as e:
        print("读取失败:", e)

print("\n检查完成")

共找到 0 个 DICOM 文件


检查完成


下面的代码用处是把所有的dcm文件转换成jpg文件，并且保留原来的目录结构, 
这个代码只处理dcm格式，不会把原本就是jpg格式的图片按照原来的结构复制到新地址，废弃

In [6]:
import os
import pydicom
import numpy as np
from PIL import Image
from tqdm import tqdm

def dcm_to_jpg_keep_structure(src_root, dst_root):
    os.makedirs(dst_root, exist_ok=True)

    dcm_files = []
    for root, _, files in os.walk(src_root):
        for file in files:
            if file.lower().endswith(".dcm"):
                dcm_files.append(os.path.join(root, file))

    print(f"共找到 {len(dcm_files)} 个 DICOM 文件")

    failed_files = []

    for dcm_path in tqdm(dcm_files):
        try:
            rel_path = os.path.relpath(dcm_path, src_root)
            rel_jpg_path = os.path.splitext(rel_path)[0] + ".jpg"
            jpg_path = os.path.join(dst_root, rel_jpg_path)

            os.makedirs(os.path.dirname(jpg_path), exist_ok=True)

            ds = pydicom.dcmread(dcm_path)
            img = ds.pixel_array
            img = np.asarray(img)

            # 情况1：单帧彩色图，且已经是 uint8
            if img.ndim == 3 and img.shape[-1] == 3 and img.dtype == np.uint8:
                pil_img = Image.fromarray(img)

            # 情况2：单帧灰度图
            elif img.ndim == 2:
                img = img.astype(np.float32)
                img = img - img.min()
                if img.max() > 0:
                    img = img / img.max()
                img = (img * 255).clip(0, 255).astype(np.uint8)
                pil_img = Image.fromarray(img)

            # 情况3：其他格式，先尽量 squeeze 再处理
            else:
                img = np.squeeze(img)

                if img.ndim == 2:
                    img = img.astype(np.float32)
                    img = img - img.min()
                    if img.max() > 0:
                        img = img / img.max()
                    img = (img * 255).clip(0, 255).astype(np.uint8)
                    pil_img = Image.fromarray(img)

                elif img.ndim == 3 and img.shape[-1] == 3:
                    if img.dtype != np.uint8:
                        img = img.astype(np.float32)
                        img = img - img.min()
                        if img.max() > 0:
                            img = img / img.max()
                        img = (img * 255).clip(0, 255).astype(np.uint8)
                    pil_img = Image.fromarray(img)

                else:
                    raise ValueError(f"暂不支持的图像形状: {img.shape}, dtype: {img.dtype}")

            pil_img.save(jpg_path, quality=95)

        except Exception as e:
            failed_files.append((dcm_path, str(e)))
            print(f"\n处理失败: {dcm_path}")
            print(e)

    print(f"\n转换完成，失败数量: {len(failed_files)}")
    return failed_files

下面是修改后的新代码，既可以处理dcm格式，也会把原本就是jpg格式的图片复制到新文件夹

In [2]:
import os
import shutil
import pydicom
import numpy as np
from PIL import Image
from tqdm import tqdm

def normalize_to_uint8(img):
    img = np.asarray(img)

    if img.ndim == 2:
        img = img.astype(np.float32)
        img = img - img.min()
        if img.max() > 0:
            img = img / img.max()
        img = (img * 255).clip(0, 255).astype(np.uint8)
        return img

    elif img.ndim == 3 and img.shape[-1] == 3:
        if img.dtype != np.uint8:
            img = img.astype(np.float32)
            img = img - img.min()
            if img.max() > 0:
                img = img / img.max()
            img = (img * 255).clip(0, 255).astype(np.uint8)
        return img

    else:
        raise ValueError(f"暂不支持的图像形状: {img.shape}, dtype: {img.dtype}")


def convert_and_copy_images(src_root, dst_root):
    os.makedirs(dst_root, exist_ok=True)

    supported_image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    dcm_files = []
    image_files = []

    for root, _, files in os.walk(src_root):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            full_path = os.path.join(root, file)

            if ext == ".dcm":
                dcm_files.append(full_path)
            elif ext in supported_image_exts:
                image_files.append(full_path)

    print(f"共找到 {len(dcm_files)} 个 DICOM 文件")
    print(f"共找到 {len(image_files)} 个普通图片文件")

    failed_files = []

    # 1. 处理 DICOM -> JPG
    for dcm_path in tqdm(dcm_files, desc="转换 DICOM"):
        try:
            rel_path = os.path.relpath(dcm_path, src_root)
            rel_jpg_path = os.path.splitext(rel_path)[0] + ".jpg"
            jpg_path = os.path.join(dst_root, rel_jpg_path)

            os.makedirs(os.path.dirname(jpg_path), exist_ok=True)

            ds = pydicom.dcmread(dcm_path)
            img = ds.pixel_array
            img = np.squeeze(img)

            # 单帧彩色 uint8
            if img.ndim == 3 and img.shape[-1] == 3 and img.dtype == np.uint8:
                pil_img = Image.fromarray(img)
            else:
                img = normalize_to_uint8(img)
                pil_img = Image.fromarray(img)

            pil_img.save(jpg_path, quality=95)

        except Exception as e:
            failed_files.append((dcm_path, str(e)))
            print(f"\n处理失败: {dcm_path}")
            print(e)

    # 2. 复制原有图片
    for img_path in tqdm(image_files, desc="复制普通图片"):
        try:
            rel_path = os.path.relpath(img_path, src_root)
            dst_path = os.path.join(dst_root, rel_path)

            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(img_path, dst_path)

        except Exception as e:
            failed_files.append((img_path, str(e)))
            print(f"\n复制失败: {img_path}")
            print(e)

    print(f"\n处理完成，失败数量: {len(failed_files)}")
    return failed_files

In [ ]:
src_root = r"G:\wxy_codes\Fetal_hydronephrosis\前期数据20260320\图像"
dst_root = r"G:\wxy_codes\Fetal_hydronephrosis\前期数据20260320\图像jpg"

failed_files = convert_and_copy_images(src_root, dst_root)

In [ ]:
##删除所有名字是0的文件夹层级，把里面的jpg文件向上提一级
import os
import shutil
from pathlib import Path

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\前期数据20260320\图像jpg_删除拆分")

moved_count = 0
removed_count = 0
skipped_count = 0

for current_root, dirs, files in os.walk(root_dir, topdown=False):
    current_path = Path(current_root)
    
    if current_path.name == "0":
        parent_dir = current_path.parent
        
        print(f"\n处理文件夹: {current_path}")
        print(f"上一级目录: {parent_dir}")
        
        for item in current_path.iterdir():
            target_path = parent_dir / item.name
            
            if target_path.exists():
                print(f"  跳过，目标已存在: {target_path}")
                skipped_count += 1
                continue
            
            shutil.move(str(item), str(target_path))
            print(f"  已移动: {item} -> {target_path}")
            moved_count += 1
        
        try:
            current_path.rmdir()
            print(f"  已删除空文件夹: {current_path}")
            removed_count += 1
        except Exception as e:
            print(f"  删除失败: {current_path}, 原因: {e}")

print("\n处理完成")
print(f"共移动文件数: {moved_count}")
print(f"共删除文件夹数: {removed_count}")
print(f"共跳过文件数: {skipped_count}")

In [ ]:
##跟上面的代码功能相反，这个是在jpg文件上级添加一个images文件夹
import shutil
from pathlib import Path

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\前期数据20260320\图像jpg_删除拆分")
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

created_images_dir_count = 0
moved_count = 0
skipped_count = 0
case_dir_count = 0

# 遍历到“病例号这一层”
# root_dir / 阳性数据或阴性数据 / 人名目录 / 病例号目录
for label_dir in root_dir.iterdir():
    if not label_dir.is_dir():
        continue

    for person_dir in label_dir.iterdir():
        if not person_dir.is_dir():
            continue

        for case_dir in person_dir.iterdir():
            if not case_dir.is_dir():
                continue

            case_dir_count += 1
            print(f"\n处理病例目录: {case_dir}")

            images_dir = case_dir / "images"

            # 找当前病例目录下的图片文件（只找当前层，不进子目录）
            img_files = [
                f for f in case_dir.iterdir()
                if f.is_file() and f.suffix.lower() in img_exts
            ]

            if not img_files:
                print("  当前目录下没有图片，跳过")
                continue

            if not images_dir.exists():
                images_dir.mkdir()
                created_images_dir_count += 1
                print(f"  已创建文件夹: {images_dir}")

            for img_file in img_files:
                target_path = images_dir / img_file.name

                if target_path.exists():
                    print(f"  跳过（目标已存在）: {target_path}")
                    skipped_count += 1
                    continue

                shutil.move(str(img_file), str(target_path))
                print(f"  已移动: {img_file.name} -> {target_path}")
                moved_count += 1

print("\n处理完成")
print(f"共扫描病例目录数: {case_dir_count}")
print(f"共创建 images 文件夹数: {created_images_dir_count}")
print(f"共移动文件数: {moved_count}")
print(f"共跳过文件数: {skipped_count}")